[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/estimation/multiview/multiview_reconstruction.ipynb)

# Multi-View Reconstruction and Camera Alignment

This example recovers where things are from several pictures of them. Three cameras look at the same handful of landmarks, each from its own place, and each picture only says in which direction a landmark lies, not how far away. Where the directions from different cameras cross, the landmark is; and when one camera's pose is a little off, its directions miss the others', and the misses say how to move the camera.

A pixel's uncertainty is modelled as a quadric on the sensor, and the camera, a map from scene points to image points, carries it back into the scene as a sight cone: `camera.adjugate()(disc(camera))`, which exists although the camera loses a dimension. Sight cones add, so the cones of one landmark from all cameras fuse into a small quadric around it, whose centre is the landmark. The cone's value at a point is the cost, and its curvature over each camera's small rigid steps gives the step that aligns the camera.

In the matrix notation of projective vision, with a camera matrix $P$ acting on homogeneous points, planes pull back through $P^\top$, and a rank-one sensor precision $n n^\top$ lifts to the sight-cone matrix $P^\top n\, n^\top P$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from itertools import islice

import numpy as np

from numga import stack
from numga.algebras import PGA2D
from examples import instantiate
from examples.animation import save_figure, show_animation
from examples.estimation.multiview import render

np.set_printoptions(precision=4, suppress=True)

# The multiview mathematics is written for any PGA; this notebook works in the plane.
core = instantiate("examples.estimation.multiview.core", PGA2D)
mv = core.mv
Scalar = core.Scalar
# Points are antivectors and planes, the lines of the plane, vectors; a motor moves rigidly, and a twist
# is a small rigid step.
Point = core.Point
Plane = core.Plane
Motor = core.Motor
Twist = core.Twist
# A camera maps scene points to image points; a quadric sends a point to its polar plane; information is
# the cost's curvature over pairs of steps.
Camera = core.Camera            # Point <- Point
Quadric = core.Quadric          # Plane <- Point
Information = core.Information  # Scalar <- (Twist, Twist)
# The plane at infinity.
w = core.w                      # [] Plane

## 1. The landmarks and the cameras

Six landmarks stand in front of three cameras: one on the left and one on the right, both turned inwards, and one between them looking straight ahead. Every camera is the same pinhole camera, carried to its place by its pose.

A pinhole camera sends a scene point to where the line from the point through the pinhole meets the sensor: the join of the pinhole with the point, met with the sensor line, `(pinhole & point) ^ sensor`. Its action on every point, `(pinhole & Point) ^ sensor`, is the camera itself: a map from scene points to image points.

In [ ]:
# The landmarks, in front of the cameras.
landmark_coordinates = [[0.15, 0.85], [-0.43, 1.15], [0.50, 1.50], [0.03, 1.85], [0.65, 2.20], [-0.60, 2.55]]
# Each camera's offset along x and how far it is turned inwards: left, right, and in the middle.
camera_offsets = np.array([-0.75, 0.75, 0.0])
camera_turns = np.radians([18.0, -18.0, 0.0])

In [ ]:
landmarks = mv.point(landmark_coordinates)                                     # [landmarks] Point
# Each camera moved along x, then turned.
rig = (mv.xw * (camera_offsets / 2)).exp() * (mv.xy * (camera_turns / 2)).exp()  # [cameras] Motor

# The pinhole at the origin and the sensor, the line one unit in front of it.
pinhole = mv.point([0.0, 0.0])                                                  # [] Point
sensor = mv.y - mv.w                                                            # [] Plane
camera = (pinhole & Point) ^ sensor                                             # [] Point <- Point
save_figure(render.draw_rig(rig), "multiview_rig");

## 2. Pixels and sight cones

Each camera sees each landmark at a pixel on its sensor. The pixel is uncertain, so it is kept as a cost around it: the squared distance from the line through it across the sensor, a quadric on the sensor's points written as the map sending a point to its polar line, `across * (across & Point)`. A quadric's value at a point is its polar joined with the point.

Feeding the camera into that quadric and carrying its polar lines back through the camera lifts it into a sight cone on scene points, wider the further from the camera. What carries lines back is the camera's adjugate, the map on lines that incidence carries over from the camera: `back_projection(line) & point == line & camera(point)` for every line and point, and it exists although the camera loses a dimension. A cone moves into the world like any map, `rig >> cones(rig << Point)`.

In the matrix notation of projective vision the adjugate reads as the transpose $P^\top$ of the camera matrix, through which lines back-project.

In [ ]:
# Every landmark as each camera sees it, in the camera's own frame, and its image on the sensor.
seen = rig << landmarks[:, None]                                                # [landmarks, cameras] Point
images = camera(seen)                                                           # [landmarks, cameras] Point
pixels = images / (w & images)                                                  # [landmarks, cameras] Point

# The line through each pixel across the sensor, and the squared distance from it as a quadric.
across = mv.x - mv.w * (mv.x & pixels)                                          # [landmarks, cameras] Plane
discs = across * (across & Point)                                               # [landmarks, cameras] Quadric

# The camera fed into each disc, and its polar lines carried back: a sight cone in each camera's frame,
# and in the world.
back_projection = camera.adjugate()                                             # [] Plane <- Plane
cones = back_projection(discs(camera))                                          # [landmarks, cameras] Quadric
world_cones = rig >> cones(rig << Point)                                        # [landmarks, cameras] Quadric
save_figure(render.draw_cones(rig, world_cones), "multiview_cones");

In [ ]:
print(back_projection)

## 3. Landmarks in closed form

The cones of one landmark from all cameras add into one quadric, a splat: small where the sight lines cross steeply, long where they are nearly parallel. Its centre is the landmark. At its centre the splat's polar vanishes, and adding a dyad on the plane at infinity, `w * (w & Point)`, makes the centre the pole of that plane, so the landmark is `(splat + w * (w & Point)).solve(w)`.

To see the cameras aligned, one of them is knocked out of place first: the cones it contributes no longer cross the others' at the landmarks, and the splats spread.

In [ ]:
# The right camera knocked out of place: moved 8 cm along x and 4 cm along y, and turned 4 degrees.
knock_offsets = np.array([0.08, -0.04])
knock_turn = np.radians(-4.0)

In [ ]:
def triangulate(poses: Motor, cones: Quadric) -> tuple[Point, Quadric]:
    """The landmarks as the centres of their splats, and the splats: each landmark's cones, carried into the
    world by the poses and added."""
    splats = (poses >> cones(poses << Point)).sum(axis=-1)                      # [landmarks] Quadric
    # The splat's polar vanishes at its centre; the dyad on the plane at infinity makes the centre its pole.
    centres = (splats + w * (w & Point)).solve(w).normalized()                  # [landmarks] Point
    return centres, splats


def distance(points: Point) -> Scalar:
    """The root mean square distance of points from the landmarks: the difference of two unit points is a
    direction, its length the norm of its complement."""
    return (points - landmarks).dual().norm_squared().mean(axis=0).square_root()   # [] Scalar


knock = (((mv.xw * knock_offsets[0] - mv.yw * knock_offsets[1]) / 2).exp()
         * (mv.xy * (knock_turn / 2)).exp())                                    # [] Motor
# The rig as it is believed to be, with the right camera knocked.
guess = rig * stack([mv.rotor(), knock, mv.rotor()])                            # [cameras] Motor
guessed_landmarks, guessed_splats = triangulate(guess, cones)                   # [landmarks] Point, Quadric
save_figure(render.draw_reconstruction(guess, guess >> cones(guess << Point), guessed_splats, guessed_landmarks),
            "multiview_guess");

In [ ]:
print(f"landmarks off by {(distance(guessed_landmarks) * 100).to_array():.2f} cm with the right camera knocked")

## 4. Aligning the cameras

The cost of the scene is every landmark's cone values, summed over the cameras: the cone's polar of the landmark, joined with the landmark. A small rigid step of a camera moves the landmarks in its frame by minus their commutator with the step, so `motion = -Twist.commutator(seen)` is a map from steps to motions of the points. The polar of the motion joined with the motion is the cost's curvature over pairs of steps, and the landmark's polar joined with the motion its slope; the cone is the cost, so no residual is formed. Solving the curvature against the slope gives each camera's step. Alternating that with placing the landmarks at their splats' centres walks the knocked camera back into place. The other two cameras stay where they are: pictures alone cannot tell the whole scene and rig moved together, and the cameras held still fix where the scene is. Each step closes about a sixth of what is left, not more, because the landmarks move with the camera between steps, which a step of the camera alone does not see. The multiview core module gives two more ways to align: one joint step of cameras and landmarks together, `align_jointly`, and steps that bring each pixel's sight line through the other cameras' splat of its landmark, with no landmark placed at all, `align_rays_to_splats`.

In the matrix notation of least squares the curvature and slope read as $J^\top J$ and $J^\top r$, with $J$ the rate of the residual $r = n^\top p$ of the single-line discs; here no Jacobian is formed, and for a quadric cost with no residual behind it the same two lines still give its curvature and slope.

In [ ]:
# The cameras that move, the knocked one; the steps shown, and the duration of a frame in milliseconds.
moving = np.array([0.0, 1.0, 0.0])
n_steps = 30
frame_duration = 100

In [ ]:
def alignment(poses: Motor, cones: Quadric, moving: np.ndarray):
    """The poses, landmarks, splats and the curvature of the cost over each camera's steps, at every step of
    the alignment, without end."""
    while True:
        centres, splats = triangulate(poses, cones)
        # The landmarks in each camera's frame, and how a small step of the camera moves them.
        seen = poses << centres[:, None]                                        # [landmarks, cameras] Point
        motion = -Twist.commutator(seen)                                        # [landmarks, cameras] Point <- Twist
        # The cone's polar of the motion joined with the motion is the curvature over pairs of steps; the
        # landmark's polar joined with the motion is the slope.
        curvature = (cones(motion) & motion).sum(axis=0)                        # [cameras] Scalar <- (Twist, Twist)
        slope = (cones(seen) & motion).sum(axis=0)                              # [cameras] Scalar <- Twist
        yield poses, centres, splats, curvature
        # Each moving camera's step; the others stay where they are.
        step = curvature.solve(-slope) * moving                                 # [cameras] Twist
        poses = poses * (step / 2).exp()                                        # [cameras] Motor


def cost(poses: Motor, centres: Point) -> Scalar:
    """Every landmark's cone values, summed over the cameras."""
    seen = poses << centres[:, None]                                            # [landmarks, cameras] Point
    return (cones(seen) & seen).sum()                                           # [] Scalar


estimates = list(islice(alignment(guess, cones, moving), n_steps + 1))
show_animation(render.animate_convergence((poses, poses >> cones(poses << Point), splats, centres)
                                          for poses, centres, splats, _ in estimates),
               "multiview_reconstruction_convergence", frame_duration)

In [ ]:
for index, (poses, centres, _, _) in islice(enumerate(estimates), None, None, 5):
    print(f"step {index}: landmarks off by {(distance(centres) * 100).to_array():.3f} cm, cost {cost(poses, centres).to_array():.3e}")

## 5. The cameras' pose uncertainty

The curvature of the cost over a camera's steps, over the variance of a pixel, is the information on its pose: how sharply the likelihood falls as the camera moves each way. Its inverse is the pose's covariance, drawn as the one-sigma region of the camera's position and the spread of its heading. Each camera's own curvature, with the others held in place, goes on the diagonal of the information on all poses together, a form over the field of camera steps.

In statistical terms the information reads as the Fisher information $\mathcal{I}$, and the covariance as $\mathcal{I}^{-1}$.

In [ ]:
# The pixel's noise, one standard deviation on the sensor one unit in front of the pinhole: as wide as the
# cones are drawn.
pixel_noise = 0.035

In [ ]:
poses, centres, splats, curvature = estimates[-1]
# Each camera's own curvature over the pixel's variance on the diagonal: a form over the field of camera steps.
information = (curvature[:, None] * np.eye(len(curvature)) / pixel_noise**2).field(1, 2)                # Scalar <- (Twist[cameras], Twist[cameras])
save_figure(render.draw_reconstruction_with_covariance(poses, poses >> cones(poses << Point), splats, centres,
                                                        information), "multiview_covariance");

In [ ]:
# One sigma of each camera's pose, along x and y and in its heading.
sigmas = render.pose_sigmas(information)                                        # [cameras, 3]
for name, (along_x, along_y, heading) in zip(("left", "right", "middle"), sigmas):
    print(f"{name:6s} camera: {along_x * 100:.3f} cm along x, {along_y * 100:.3f} cm along y, {np.degrees(heading):.3f} degrees")

In [ ]:
# checks
# The landmarks seen from the true rig are found exactly; the knocked camera spreads them; the alignment
# brings them back, and the cost falls to round-off.
true_centres, _ = triangulate(rig, cones)
np.testing.assert_allclose(distance(true_centres).to_array(), 0.0, atol=1e-9)
assert distance(guessed_landmarks).to_array() > 1e-3
np.testing.assert_allclose(distance(centres).to_array(), 0.0, atol=1e-3)
assert cost(*estimates[-1][:2]).to_array() < 1e-3 * cost(*estimates[0][:2]).to_array()